# Tuned SGC Model — Optuna Best Parameters

This notebook trains and evaluates the SGCBaseline model using the best hyperparameters
found by Optuna in `hyperparam_tuning.ipynb`:

| Parameter | Value |
|---|---|
| `model_type` | `sgc` |
| `use_weighted_loss` | `False` |
| `k_prop` | `2` |
| `hidden_dim` | `64` |
| `mlp_dim` | `512` |
| `dropout` | `0.2847` |
| `weight_decay` | `0.007024` |
| `patience` | `20` |

Runs 3-fold CV on training data, then generates fold-averaged and seed-averaged test submissions.

## Imports

In [ ]:
!pip install -r requirements.txt -q
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu118
!pip install accelerate -U

In [ ]:
%pip install torch_geometric

## Setup — Imports, Constants & Tuned Hyperparameters

In [ ]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
from scipy.spatial.distance import jensenshannon
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

from utils.MatrixVectorizer import MatrixVectorizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DATA_DIR      = "data"
TRAIN_LR_PATH = os.path.join(DATA_DIR, "lr_train.csv")
TRAIN_HR_PATH = os.path.join(DATA_DIR, "hr_train.csv")
TEST_LR_PATH  = os.path.join(DATA_DIR, "lr_test.csv")

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Fixed training settings ──────────────────────────────────────────────────
EPOCHS     = 200
LR         = 1e-3
BATCH_SIZE = 16

# ── Optuna best parameters ───────────────────────────────────────────────────
MODEL_TYPE        = "sgc"
USE_WEIGHTED_LOSS = False
K_PROP            = 2
HIDDEN_DIM        = 64
MLP_DIM           = 512
DROPOUT           = 0.2846780493863432
WEIGHT_DECAY      = 0.007023868736399456
PATIENCE          = 20

# Seed-averaging settings
SEEDS        = [42, 123, 999]
RUN_SEED_AVG = True

print(f"Device : {DEVICE}")
print(f"Tuned hyperparameters:")
print(f"  model_type={MODEL_TYPE}  k_prop={K_PROP}  hidden_dim={HIDDEN_DIM}  mlp_dim={MLP_DIM}")
print(f"  dropout={DROPOUT:.6f}  weight_decay={WEIGHT_DECAY:.6f}  patience={PATIENCE}")
print(f"  use_weighted_loss={USE_WEIGHTED_LOSS}")

In [ ]:
%run utils/reproducibility.py

# Sync notebook constants with reproducibility.py
DEVICE = device
RANDOM_SEED = random_seed

In [ ]:
def sanitize_vectors(x: np.ndarray) -> np.ndarray:
    x = np.nan_to_num(x)
    x[x < 0] = 0
    return x

def load_train_data(lr_path: str, hr_path: str):
    lr_vecs = pd.read_csv(lr_path).to_numpy(dtype=np.float32)
    hr_vecs = pd.read_csv(hr_path).to_numpy(dtype=np.float32)
    return sanitize_vectors(lr_vecs), sanitize_vectors(hr_vecs)

def load_test_lr(path: str) -> np.ndarray | None:
    if not os.path.exists(path):
        print(f"WARNING: test file not found: {path}. Skipping test inference.")
        return None
    return sanitize_vectors(pd.read_csv(path).to_numpy(dtype=np.float32))

def remove_outlier_subjects(lr_vecs: np.ndarray, hr_vecs: np.ndarray, n_std: float = 3.0):
    per_subject_mean = hr_vecs.mean(axis=1)
    threshold = per_subject_mean.mean() + n_std * per_subject_mean.std()
    keep = per_subject_mean <= threshold
    n_removed = int((~keep).sum())
    if n_removed:
        print(f"[outlier filter] removed {n_removed} subject(s) with mean HR > {threshold:.3f}: "
              f"indices {np.where(~keep)[0].tolist()}")
    return lr_vecs[keep], hr_vecs[keep]

def vectors_to_matrices(vectors: np.ndarray, matrix_size: int) -> np.ndarray:
    mats = [MatrixVectorizer.anti_vectorize(v, matrix_size) for v in vectors]
    return np.stack(mats, axis=0)

def normalize_adj_batch(adj: torch.Tensor) -> torch.Tensor:
    b, n, _ = adj.shape
    eye = torch.eye(n, device=adj.device).unsqueeze(0).expand(b, -1, -1)
    adj = adj + eye
    degree = adj.sum(dim=2)
    d_inv_sqrt = degree.pow(-0.5)
    d_inv_sqrt[d_inv_sqrt == float("inf")] = 0.0
    return adj * d_inv_sqrt.unsqueeze(2) * d_inv_sqrt.unsqueeze(1)

## Model Definition — SGCBaseline (tuned)

In [ ]:
class SGCBaseline(nn.Module):
    """SGC (Simple Graph Convolution) baseline.
    - SGC propagation (K steps) over LR adjacency.
    - Node-level projection → flatten → MLP → HR edge vector (length 35778).
    - Linear output; clipped to [0, 1] at inference.
    """
    def __init__(
        self,
        lr_nodes:   int   = 160,
        k_prop:     int   = 2,
        hidden_dim: int   = 64,
        mlp_dim:    int   = 512,
        out_dim:    int   = 35778,
        dropout:    float = 0.2847,
    ):
        super().__init__()
        self.lr_nodes  = lr_nodes
        self.k_prop    = k_prop
        self.node_proj = nn.Linear(lr_nodes, hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(lr_nodes * hidden_dim, mlp_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, out_dim),
        )

    def forward(self, adj_lr: torch.Tensor) -> torch.Tensor:
        adj_norm = normalize_adj_batch(adj_lr)
        x = adj_lr
        for _ in range(self.k_prop):
            x = torch.bmm(adj_norm, x)
        h = self.node_proj(x)
        h_flat = h.reshape(h.size(0), -1)
        return self.mlp(h_flat)


def build_model() -> nn.Module:
    """Instantiate SGCBaseline with Optuna-tuned hyperparameters."""
    return SGCBaseline(
        lr_nodes=160,
        k_prop=K_PROP,
        hidden_dim=HIDDEN_DIM,
        mlp_dim=MLP_DIM,
        out_dim=35778,
        dropout=DROPOUT,
    )

In [ ]:
# Training helpers

def train_epoch(model, loader, optimiser, loss_fn):
    model.train()
    running = 0.0
    for adj_lr, hr_vec in loader:
        adj_lr = adj_lr.to(DEVICE)
        hr_vec = hr_vec.to(DEVICE)
        optimiser.zero_grad()
        pred = model(adj_lr)
        loss = loss_fn(pred, hr_vec)
        loss.backward()
        optimiser.step()
        running += loss.item() * adj_lr.size(0)
    return running / len(loader.dataset)


def val_epoch(model, loader) -> float:
    model.eval()
    mae_loss = nn.L1Loss()
    total, n = 0.0, 0
    with torch.no_grad():
        for adj_lr, hr_vec in loader:
            adj_lr = adj_lr.to(DEVICE)
            hr_vec = hr_vec.to(DEVICE)
            pred   = model(adj_lr)
            total += mae_loss(pred, hr_vec).item() * adj_lr.size(0)
            n     += adj_lr.size(0)
    return total / n


def predict(model, loader) -> np.ndarray:
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            adj_lr = (batch[0] if isinstance(batch, (list, tuple)) else batch).to(DEVICE)
            preds.append(model(adj_lr).cpu().numpy())
    return np.concatenate(preds, axis=0)


def postprocess(pred_vecs: np.ndarray) -> np.ndarray:
    return np.clip(np.nan_to_num(pred_vecs), 0.0, 1.0)


def save_predictions_csv(pred_vecs: np.ndarray, out_path: str):
    flat = pred_vecs.flatten()
    pd.DataFrame({"ID": np.arange(1, len(flat) + 1, dtype=np.int32), "Predicted": flat}).to_csv(out_path, index=False)

## Data Loading & Preprocessing

In [ ]:
lr_vecs_raw, hr_vecs_raw = load_train_data(TRAIN_LR_PATH, TRAIN_HR_PATH)

assert lr_vecs_raw.shape == (167, 12720), f"Unexpected LR train shape: {lr_vecs_raw.shape}"
assert hr_vecs_raw.shape == (167, 35778), f"Unexpected HR train shape: {hr_vecs_raw.shape}"
print(f"[sanity] LR train : shape={lr_vecs_raw.shape}  range=[{lr_vecs_raw.min():.4f}, {lr_vecs_raw.max():.4f}]")
print(f"[sanity] HR train : shape={hr_vecs_raw.shape}  range=[{hr_vecs_raw.min():.4f}, {hr_vecs_raw.max():.4f}]")

lr_vecs, hr_vecs = remove_outlier_subjects(lr_vecs_raw, hr_vecs_raw)

lr_mats = vectors_to_matrices(lr_vecs, 160).astype(np.float32)
hr_vecs = hr_vecs.astype(np.float32)

print(f"LR matrices : {lr_mats.shape}")
print(f"HR vectors  : {hr_vecs.shape}")

## 3-Fold Cross-Validation Training

In [ ]:
from utils.training import compute_metrics

loss_fn = nn.L1Loss()  # use_weighted_loss=False per best params
kfold   = KFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

all_fold_metrics = []
oof_preds_list   = []
oof_gts_list     = []

print(f"Optimizer  : AdamW  lr={LR}  weight_decay={WEIGHT_DECAY:.6f}")
print(f"Early stop : patience={PATIENCE}  max_epochs={EPOCHS}")
print(f"Model      : SGCBaseline  k_prop={K_PROP}  hidden_dim={HIDDEN_DIM}  mlp_dim={MLP_DIM}  dropout={DROPOUT:.6f}\n")

for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(lr_mats), start=1):
    print(f"\n{'='*50}")
    print(f"  Fold {fold_idx}/3")
    print(f"{'='*50}")

    train_adj = torch.from_numpy(lr_mats[train_idx])
    train_hr  = torch.from_numpy(hr_vecs[train_idx])
    val_adj   = torch.from_numpy(lr_mats[val_idx])
    val_hr    = torch.from_numpy(hr_vecs[val_idx])

    def worker_init_fn(worker_id):
        np.random.seed(RANDOM_SEED + worker_id)

    train_loader = DataLoader(
        TensorDataset(train_adj, train_hr),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
        worker_init_fn=worker_init_fn,
    )
    val_loader = DataLoader(
        TensorDataset(val_adj, val_hr),
        batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    )

    model     = build_model().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_mae, best_epoch = float("inf"), 0
    patience_count, best_weights = 0, None

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn)
        v_mae      = val_epoch(model, val_loader)

        if epoch % 20 == 0:
            print(f"  Epoch {epoch:03d} | train_loss={train_loss:.6f} | val_MAE={v_mae:.6f}")

        if v_mae < best_val_mae - 1e-7:
            best_val_mae, best_epoch = v_mae, epoch
            best_weights = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).")
                break

    model.load_state_dict(best_weights)
    print(f"\n  Best epoch: {best_epoch}  |  Best val MAE: {best_val_mae:.6f}")

    preds = postprocess(predict(model, val_loader))
    oof_preds_list.append(preds)
    oof_gts_list.append(hr_vecs[val_idx])

    fold_csv = os.path.join(OUTPUT_DIR, f"predictions_fold_{fold_idx}.csv")
    save_predictions_csv(preds, fold_csv)

    metrics = compute_metrics(preds, hr_vecs[val_idx], sanitize_fn=sanitize_vectors)
    all_fold_metrics.append(metrics)

    print("\n  Fold metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.6f}")

# Summary
metric_names = list(all_fold_metrics[0].keys())
means = [np.mean([m[n] for m in all_fold_metrics]) for n in metric_names]
stds  = [np.std( [m[n] for m in all_fold_metrics]) for n in metric_names]

print("\n" + "="*50)
print("  3-Fold CV Summary (mean ± std)")
print("="*50)
for name, mu, sigma in zip(metric_names, means, stds):
    print(f"  {name:12s}: {mu:.6f} ± {sigma:.6f}")

In [ ]:
from utils.plot_chart import plot_fold_metrics

os.makedirs("figs", exist_ok=True)

fig = plot_fold_metrics(
    all_fold_metrics,
    title="Tuned SGC Model – 3-Fold Cross-Validation Metrics",
    save_path="figs/tuned_cv_metrics.png",
)
plt.show()

In [ ]:
# OOF Flatten MAE
oof_preds = np.concatenate(oof_preds_list, axis=0)
oof_gts   = np.concatenate(oof_gts_list,   axis=0)

oof_flat_mae = mean_absolute_error(oof_gts.flatten(), oof_preds.flatten())
print(f"OOF Flatten MAE (all folds concatenated): {oof_flat_mae:.6f}")
print(f"  Total subjects in OOF: {oof_preds.shape[0]}")

## Test Submissions

1. **`submission_tuned_foldavg.csv`** — single seed (42), fold-averaged test predictions.
2. **`submission_tuned_foldavg_seedavg.csv`** — averaged over `SEEDS` × 3 folds (controlled by `RUN_SEED_AVG`).

In [ ]:
def run_cv_and_get_test_preds(
    lr_mats_tr: np.ndarray,
    hr_vecs_tr:  np.ndarray,
    lr_mats_te:  np.ndarray,
    seed:        int,
) -> np.ndarray:
    """Train one full 3-fold CV for a given seed; return averaged test predictions."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    kf      = KFold(n_splits=3, shuffle=True, random_state=seed)
    loss_fn = nn.L1Loss()

    test_tensor = torch.from_numpy(lr_mats_te)
    dummy_hr    = torch.zeros(len(lr_mats_te), hr_vecs_tr.shape[1])
    test_loader = DataLoader(
        TensorDataset(test_tensor, dummy_hr),
        batch_size=BATCH_SIZE, shuffle=False,
    )

    fold_test_preds = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(lr_mats_tr), start=1):
        print(f"  [seed={seed}] Fold {fold_idx}/3 ...", end=" ", flush=True)

        train_loader = DataLoader(
            TensorDataset(
                torch.from_numpy(lr_mats_tr[train_idx]),
                torch.from_numpy(hr_vecs_tr[train_idx]),
            ),
            batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
        )
        val_loader = DataLoader(
            TensorDataset(
                torch.from_numpy(lr_mats_tr[val_idx]),
                torch.from_numpy(hr_vecs_tr[val_idx]),
            ),
            batch_size=BATCH_SIZE, shuffle=False,
        )

        model     = build_model().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val_mae, best_epoch = float("inf"), 0
        patience_count, best_weights = 0, None

        for epoch in range(1, EPOCHS + 1):
            train_epoch(model, train_loader, optimizer, loss_fn)
            v_mae = val_epoch(model, val_loader)

            if v_mae < best_val_mae - 1e-7:
                best_val_mae, best_epoch = v_mae, epoch
                best_weights = copy.deepcopy(model.state_dict())
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= PATIENCE:
                    break

        model.load_state_dict(best_weights)
        print(f"best_epoch={best_epoch}  val_MAE={best_val_mae:.6f}")

        fold_test_preds.append(postprocess(predict(model, test_loader)))

    return np.mean(fold_test_preds, axis=0)


# ── Phase 3.1: Single-seed fold-averaged test submission ─────────────────────
test_lr_vecs = load_test_lr(TEST_LR_PATH)

if test_lr_vecs is not None:
    assert test_lr_vecs.shape == (112, 12720), f"Unexpected LR test shape: {test_lr_vecs.shape}"
    print(f"[sanity] LR test  : shape={test_lr_vecs.shape}  range=[{test_lr_vecs.min():.4f}, {test_lr_vecs.max():.4f}]")

    test_lr_mats = vectors_to_matrices(test_lr_vecs, 160).astype(np.float32)
    print(f"\n[Phase 3.1] Generating fold-averaged submission  (seed={RANDOM_SEED}) ...")
    test_preds_foldavg = run_cv_and_get_test_preds(lr_mats, hr_vecs, test_lr_mats, seed=RANDOM_SEED)

    sub_path = os.path.join(OUTPUT_DIR, "submission_tuned_foldavg.csv")
    save_predictions_csv(test_preds_foldavg, sub_path)
    print(f"  Saved fold-averaged submission → {sub_path}")
    print(f"  Value range: [{test_preds_foldavg.min():.4f}, {test_preds_foldavg.max():.4f}]")
else:
    test_lr_mats       = None
    test_preds_foldavg = None
    print("[Phase 3.1] Skipped (no test file).")

# ── Phase 3.2: Multi-seed seed-averaged test submission ──────────────────────
if test_lr_mats is not None and RUN_SEED_AVG:
    print(f"\n[Phase 3.2] Seed-averaging over seeds={SEEDS} ...")
    all_seed_preds = []

    for seed in SEEDS:
        print(f"\n  Seed {seed}")
        seed_preds = run_cv_and_get_test_preds(lr_mats, hr_vecs, test_lr_mats, seed=seed)
        per_seed_path = os.path.join(OUTPUT_DIR, f"submission_tuned_seed{seed}_foldavg.csv")
        save_predictions_csv(seed_preds, per_seed_path)
        print(f"  Saved per-seed submission → {per_seed_path}")
        all_seed_preds.append(seed_preds)

    test_preds_seedavg = postprocess(np.mean(all_seed_preds, axis=0))
    seedavg_path = os.path.join(OUTPUT_DIR, "submission_tuned_foldavg_seedavg.csv")
    save_predictions_csv(test_preds_seedavg, seedavg_path)
    print(f"\n  Saved seed-averaged submission → {seedavg_path}")
    print(f"  Averaged over {len(SEEDS)} seeds × 3 folds = {len(SEEDS)*3} models.")
    print(f"  Value range: [{test_preds_seedavg.min():.4f}, {test_preds_seedavg.max():.4f}]")

elif test_lr_mats is None:
    print("[Phase 3.2] Skipped (no test file).")
else:
    print("[Phase 3.2] Skipped (RUN_SEED_AVG=False).")

print("\nDone. Files written to outputs/:")
for fname in sorted(f for f in os.listdir(OUTPUT_DIR) if "tuned" in f):
    path = os.path.join(OUTPUT_DIR, fname)
    print(f"  {fname:<55s}  ({os.path.getsize(path)/1024:.1f} KB)")